In [2]:
#libraries 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
home = os.path.expanduser("~")
import requests
import time

In [3]:
df_combined = pd.read_csv("/Users/hinahaq/Downloads/Book_recomendation_Project/Books_combined.csv")
print(df_combined.shape)
print("\nMissing values:")
print(df_combined.isnull().sum())
print("\nMissing %:")
print((df_combined.isnull().sum() / len(df_combined) * 100).round(1))

(2004, 18)

Missing values:
rank              1000
title                0
author               1
avg_rating         755
num_ratings        733
cover_img_url        4
goodreads_url     1000
source               0
year_published       7
genres               7
description          0
pages               19
series            1529
isbn              1008
publisher         1009
language          1004
ol_key            1004
google_url        1017
dtype: int64

Missing %:
rank              49.9
title              0.0
author             0.0
avg_rating        37.7
num_ratings       36.6
cover_img_url      0.2
goodreads_url     49.9
source             0.0
year_published     0.3
genres             0.3
description        0.0
pages              0.9
series            76.3
isbn              50.3
publisher         50.3
language          50.1
ol_key            50.1
google_url        50.7
dtype: float64


In [4]:
# Find the 4 missing cover rows
missing_covers = df_combined[df_combined["cover_img_url"].isna()]
print(missing_covers[["title", "ol_key", "isbn"]].to_string())

                                title              ol_key           isbn
1240               The business world  /works/OL19729600M  9781501500725
1548               Raiders in Kashmir   /works/OL6656468W     9693600258
1584  Principles of political science   /works/OL5558230W            NaN
1957      Indian sociological thought   /works/OL4261520W  9788131602263


In [12]:
#fetching the cover image urls
# Open Library cover URL can be built from ol_key or isbn

for idx in [1240, 1548, 1584, 1957]:
    ol_key = df_combined.at[idx, "ol_key"]
    olid = ol_key.replace("/works/", "")
    cover_url = f"https://covers.openlibrary.org/b/olid/{olid}-L.jpg"
    df_combined.at[idx, "cover_img_url"] = cover_url
    print(f"  ✓ {df_combined.at[idx, 'title']} → {cover_url}")

df_combined.to_csv(os.path.join(home, "books_combined.csv"), index=False)
print("\nSaved!")
print(f"Missing cover images: {df_combined['cover_img_url'].isna().sum()}")

  ✓ The business world → https://covers.openlibrary.org/b/olid/OL19729600M-L.jpg
  ✓ Raiders in Kashmir → https://covers.openlibrary.org/b/olid/OL6656468W-L.jpg
  ✓ Principles of political science → https://covers.openlibrary.org/b/olid/OL5558230W-L.jpg
  ✓ Indian sociological thought → https://covers.openlibrary.org/b/olid/OL4261520W-L.jpg

Saved!
Missing cover images: 0


In [13]:
#check

print(df_combined.isnull().sum())
print(f"\nShape: {df_combined.shape}")

rank              1000
title                0
author               1
avg_rating         755
num_ratings        733
cover_img_url        0
goodreads_url     1000
source               0
year_published       7
genres               7
description          0
pages               19
series            1529
isbn              1008
publisher         1009
language          1004
ol_key            1004
google_url        1017
dtype: int64

Shape: (2004, 18)


In [14]:
#check missing author
print(df_combined[df_combined["author"].isna()][["title", "source", "ol_key"]])

                   title        source              ol_key
1240  The business world  Open Library  /works/OL19729600M


In [21]:
df_combined = df_combined.reset_index(drop=True)
df_combined["book_id"] = range(1, len(df_combined) + 1)

# Move book_id to first column
cols = ["book_id"] + [c for c in df_combined.columns if c != "book_id"]
df_combined = df_combined[cols]

df_combined.to_csv(os.path.join(home, "books_combined.csv"), index=False)
print(f"Saved! Shape: {df_combined.shape}")
print(df_combined[["book_id", "title"]].head(3))
print(df_combined[["book_id", "title"]].iloc[1003:1006])

Saved! Shape: (2004, 19)
   book_id                                    title
0        1  The Hunger Games (The Hunger Games, #1)
1        2                      Pride and Prejudice
2        3                    To Kill a Mockingbird
      book_id                                    title
1003     1004  The Strength of the Few (Hierarchy, #2)
1004     1005                            Wings of Fire
1005     1006    The Diary of a Young Girl- Anne Frank


In [5]:
# Check a GoodReads row vs Open Library row side by side
print("=== GOODREADS ROW ===")
print(df_combined.iloc[0][["title", "author", "year_published", "pages", "genres"]].to_string())

print("\n=== OPEN LIBRARY ROW ===")
print(df_combined.iloc[1005][["title", "author", "year_published", "pages", "genres"]].to_string())

=== GOODREADS ROW ===
title                       The Hunger Games (The Hunger Games, #1)
author                                              Suzanne Collins
year_published                   First published September 14, 2008
pages                                          374 pages, Hardcover
genres            Young Adult, Dystopia, Fiction, Fantasy, Scien...

=== OPEN LIBRARY ROW ===
title             The Diary of a Young Girl- Anne Frank
author                                       Anne Frank
year_published                                   2001.0
pages                                             320.0
genres                            Historical, Biography


In [6]:
import re

# Fix year_published — extract just the 4-digit year
def clean_year(val):
    if pd.isna(val):
        return None
    val = str(val)
    match = re.search(r'\b(1[0-9]{3}|20[0-9]{2})\b', val)
    if match:
        return int(match.group(1))
    return None

# Fix pages — extract just the number
def clean_pages(val):
    if pd.isna(val):
        return None
    val = str(val)
    match = re.search(r'\d+', val)
    if match:
        return int(match.group())
    return None

df_combined["year_published"] = df_combined["year_published"].apply(clean_year)
df_combined["pages"] = df_combined["pages"].apply(clean_pages)

# Verify
print("=== GOODREADS ROW ===")
print(df_combined.iloc[0][["title", "year_published", "pages"]])
print("\n=== OPEN LIBRARY ROW ===")
print(df_combined.iloc[1005][["title", "year_published", "pages"]])

# Save
df_combined.to_csv(os.path.join(home, "books_combined.csv"), index=False)
print("\nSaved!")

=== GOODREADS ROW ===
title             The Hunger Games (The Hunger Games, #1)
year_published                                     2008.0
pages                                               374.0
Name: 0, dtype: object

=== OPEN LIBRARY ROW ===
title             The Diary of a Young Girl- Anne Frank
year_published                                   2001.0
pages                                             320.0
Name: 1005, dtype: object

Saved!


In [7]:
#float to integer
df_combined["year_published"] = df_combined["year_published"].astype("Int64")
df_combined["pages"] = df_combined["pages"].astype("Int64")

# Verify
print(df_combined.iloc[0][["title", "year_published", "pages"]])
print(df_combined.iloc[1005][["title", "year_published", "pages"]])

# Save
df_combined.to_csv(os.path.join(home, "books_combined.csv"), index=False)
print("Saved!")

title             The Hunger Games (The Hunger Games, #1)
year_published                                       2008
pages                                                 374
Name: 0, dtype: object
title             The Diary of a Young Girl- Anne Frank
year_published                                     2001
pages                                               320
Name: 1005, dtype: object
Saved!
